<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 05 · Unsupervised Overview

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
Chapter 7 is intentionally light. The goal is to make clustering and PCA
feel familiar enough that the reader can recognize when they are useful.

This cell prepares the preprocessing steps so the model can work with both numeric and categorical inputs.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
from pathlib import Path
import importlib.util
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


This cell prepares the preprocessing steps so the model can work with both numeric and categorical inputs.


In [ ]:
from pathlib import Path
import importlib.util

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

df = churn_report.load_customers()
numeric = df[churn_report.NUMERIC_COLUMNS]

scaled = StandardScaler().fit_transform(numeric)
pca = PCA(n_components=2, random_state=0)
reduced = pca.fit_transform(scaled)
labels = KMeans(n_clusters=2, random_state=0, n_init=10).fit_predict(scaled)

summary = pd.DataFrame(
    reduced,
    columns=['pc1', 'pc2'],
    index=df.index,
)
summary['cluster'] = labels
summary['customer_id'] = df[churn_report.ID_COLUMN].values

print('explained variance ratio =', pca.explained_variance_ratio_)
print()
print(summary.to_string(index=False))
print()
print(df.assign(cluster=labels).groupby('cluster')[churn_report.NUMERIC_COLUMNS].mean().round(2).to_string())


### Where We Are Heading Next
Chapter 8 turns the same care and skepticism into a capstone evaluation
report.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
